# Iceberg vs Delta Lake vs Hudi: full benchmark (self-contained Colab)
**Nothing to upload.** All scripts are written by the cells below. Run the cells **top to bottom**.

What it does: generates TPC-DS `store_sales` (5 years of order lines) with DuckDB, loads it into Iceberg, Delta Lake and Hudi under the same Spark 3.5, then measures load, query latency, upserts, compaction, storage cost, time travel, schema evolution and training-set reproducibility. It finishes with detailed tables, charts, failure details, a weighted scorecard, an auto-generated recommendation and a downloadable `report.html`.

**Before you run:** Runtime → Change runtime type → **High-RAM** if available. Default scale factor is 1 (about 2.9M order lines). Expect roughly **30-90 minutes** (first use downloads connector JARs). If Colab disconnects, re-run from the top; finished steps are skipped.

In [ ]:
#@title 1. Environment (Java 17 + isolated Python 3.11 with PySpark 3.5.3)
import os, glob, subprocess, platform
ROOT = "/content/lakehouse-bench"; PY = "/content/venv/bin/python"
os.makedirs(f"{ROOT}/scripts", exist_ok=True); os.makedirs(f"{ROOT}/results/raw", exist_ok=True); os.makedirs(f"{ROOT}/site", exist_ok=True)
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null 2>&1
os.environ["JAVA_HOME"] = glob.glob("/usr/lib/jvm/java-17-openjdk*")[0]
!pip -q install uv 2>&1 | tail -1
!uv venv /content/venv --python 3.11 -q
!uv pip install -q --python /content/venv/bin/python pyspark==3.5.3 delta-spark==3.2.0 duckdb pandas
env = subprocess.run("echo CPUs: $(nproc); free -g | sed -n 2p; df -h /content | tail -1; $JAVA_HOME/bin/java -version 2>&1 | head -1; /content/venv/bin/python -c 'import pyspark;print(\"pyspark\",pyspark.__version__)'", shell=True, capture_output=True, text=True, env=os.environ).stdout
open(f"{ROOT}/results/env.txt", "w").write(env); print(env)

## 2. Benchmark code (written to disk; expand to read it, no need to edit)

In [ ]:
%%writefile /content/lakehouse-bench/config.py
"""Central benchmark configuration. Override with environment variables."""
import os
SF            = int(os.getenv("TPCDS_SF", "10"))                 # 10 .. 1000
ROOT          = os.getenv("BENCH_ROOT", os.path.abspath("./bench_data"))   # local path or s3a://bucket/prefix
DATA_DIR      = f"{ROOT}/tpcds_sf{SF}/source"                    # raw TPC-DS parquet (written by 00_generate_data.py)
WAREHOUSE     = f"{ROOT}/tpcds_sf{SF}/warehouse"                 # one sub-dir per format
RESULTS_DIR   = os.path.abspath("./results/raw")
FORMATS       = ["iceberg", "delta", "hudi"]
TARGET_FILE_BYTES = 128 * 1024 * 1024                            # same target file size for all formats
TABLE_MODE    = os.getenv("TABLE_MODE", "cow")                   # cow | mor  (copy-on-write vs merge-on-read where supported)
SPARK_MASTER  = os.getenv("SPARK_MASTER", "local[*]")
DRIVER_MEM    = os.getenv("DRIVER_MEM", "16g")
WARM_RUNS     = 3                                                # + 1 cold run per query
UPSERT_FRACTIONS = [0.001, 0.01, 0.05]                           # share of table rows per batch
UPSERT_SCENARIOS = ["recent", "scattered"]                       # late-arriving corrections vs. random history edits
FRAGMENT_BATCHES = 20                                            # tiny commits used to create small files before compaction
S3_USD_PER_GB_MONTH = 0.023                                      # storage price used for the cost model
KEY           = ["ss_item_sk", "ss_ticket_number"]               # business key (order line)
PARTITION_COL = "sold_ym"                                        # derived YYYY-MM  -> ~60 partitions over 5 years
SHUFFLE_PARTS = os.getenv("SHUFFLE_PARTS", "200")                     # lower (e.g. 8) for small local runs

In [ ]:
%%writefile /content/lakehouse-bench/scripts/common.py
"""Spark session factory + one thin adapter per table format so every test runs the *same* logic."""
import os, sys, json, time, datetime as dt, argparse
from contextlib import contextmanager
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import config as C
from pyspark.sql import SparkSession, functions as F

PKGS = {
    "iceberg": "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
    "delta":   "io.delta:delta-spark_2.12:3.2.0",
    "hudi":    "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0",
}
VERSIONS = {"iceberg": "1.6.1", "delta": "3.2.0", "hudi": "0.15.0", "spark": "3.5.x"}
COND = " AND ".join(f"t.{k}=s.{k}" for k in C.KEY)

def get_spark(fmt):
    b = (SparkSession.builder.appName(f"bench-{fmt}").master(C.SPARK_MASTER)
         .config("spark.jars.packages", PKGS[fmt])
         .config("spark.driver.memory", C.DRIVER_MEM)
         .config("spark.driver.extraJavaOptions", "-Duser.timezone=UTC")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.shuffle.partitions", C.SHUFFLE_PARTS))
    if fmt == "iceberg":
        b = (b.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
              .config("spark.sql.catalog.ice", "org.apache.iceberg.spark.SparkCatalog")
              .config("spark.sql.catalog.ice.type", "hadoop")
              .config("spark.sql.catalog.ice.warehouse", f"{C.WAREHOUSE}/iceberg")
              .config("spark.sql.catalog.ice.cache-enabled", "false"))
    elif fmt == "delta":
        b = (b.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
              .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
              .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")
              .config("spark.databricks.delta.optimize.maxFileSize", str(C.TARGET_FILE_BYTES)))
    elif fmt == "hudi":
        b = (b.config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
              .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
              .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
              .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar"))
    s = b.getOrCreate(); s.sparkContext.setLogLevel("WARN"); return s

# ---------- measurement helpers ----------
@contextmanager
def timed():
    r = {}; t = time.perf_counter()
    yield r
    r["seconds"] = round(time.perf_counter() - t, 3)

def record(fmt, test, metric, value, unit="", **detail):
    os.makedirs(C.RESULTS_DIR, exist_ok=True)
    row = {"format": fmt, "test": test, "metric": metric, "value": value, "unit": unit, "detail": detail,
           "sf": C.SF, "mode": C.TABLE_MODE, "versions": VERSIONS, "ts": dt.datetime.utcnow().isoformat() + "Z"}
    with open(f"{C.RESULTS_DIR}/{test}_{fmt}.jsonl", "a") as f: f.write(json.dumps(row, default=str) + "\n")
    print(f"[{fmt}] {test}.{metric} = {value} {unit}")

def state_path(fmt): return os.path.join(C.RESULTS_DIR, "..", f"state_{fmt}.json")
def state_get(fmt):
    p = state_path(fmt); return json.load(open(p)) if os.path.exists(p) else {}
def state_put(fmt, **kw):
    s = state_get(fmt); s.update(kw); json.dump(s, open(state_path(fmt), "w"), default=str)

def _fs(spark, path):
    p = spark._jvm.org.apache.hadoop.fs.Path(path); return p, p.getFileSystem(spark._jsc.hadoopConfiguration())
def dir_bytes(spark, path):
    p, fs = _fs(spark, path); return fs.getContentSummary(p).getLength() if fs.exists(p) else 0
def data_files(spark, path):
    p, fs = _fs(spark, path); it = fs.listFiles(p, True); n = 0
    while it.hasNext():
        s = it.next().getPath().toString()
        if s.endswith(".parquet") and "/.hoodie/" not in s and "/metadata/" not in s: n += 1
    return n
def meta_bytes(spark, path):
    return sum(dir_bytes(spark, f"{path}/{d}") for d in ("metadata", "_delta_log", ".hoodie"))

def fingerprint(df):
    """Order-independent content hash of a training window: row count + sum of per-row xxhash64."""
    r = df.select(F.count("*").alias("c"),
                  F.sum(F.xxhash64("ss_item_sk", "ss_ticket_number", "ss_quantity", "ss_net_paid").cast("decimal(38,0)")).alias("h")).first()
    return {"rows": r["c"], "hash": str(r["h"])}

# ---------- adapters ----------
class Base:
    def __init__(s, spark, fmt):
        s.spark, s.fmt = spark, fmt
        s.path = f"{C.WAREHOUSE}/{fmt}/store_sales"
    def root(s): return f"{C.WAREHOUSE}/iceberg/bench/store_sales" if s.fmt == "iceberg" else s.path
    def size(s): return dir_bytes(s.spark, s.root())
    def files(s): return data_files(s.spark, s.root())
    def meta(s): return meta_bytes(s.spark, s.root())
    def view(s, name="fact", **tt):
        s.read(**tt).createOrReplaceTempView(name); return name

class Iceberg(Base):
    ID = "ice.bench.store_sales"
    def write_initial(s, df):
        s.spark.sql("CREATE NAMESPACE IF NOT EXISTS ice.bench"); s.spark.sql(f"DROP TABLE IF EXISTS {s.ID} PURGE")
        mode = "merge-on-read" if C.TABLE_MODE == "mor" else "copy-on-write"
        (df.sortWithinPartitions(C.PARTITION_COL).writeTo(s.ID).using("iceberg").partitionedBy(F.col(C.PARTITION_COL))
           .tableProperty("format-version", "2").tableProperty("write.target-file-size-bytes", str(C.TARGET_FILE_BYTES))
           .tableProperty("write.update.mode", mode).tableProperty("write.merge.mode", mode).tableProperty("write.delete.mode", mode)
           .tableProperty("write.spark.fanout.enabled", "true").create())
    def read(s, ref=None, ts_ms=None):
        r = s.spark.read
        if ref is not None: r = r.option("snapshot-id", ref)
        if ts_ms is not None: r = r.option("as-of-timestamp", ts_ms)
        return r.table(s.ID)
    def upsert(s, src):
        src.createOrReplaceTempView("src")
        s.spark.sql(f"MERGE INTO {s.ID} t USING src s ON {COND} WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    def delete(s, rows):
        rows.createOrReplaceTempView("src")
        s.spark.sql(f"MERGE INTO {s.ID} t USING src s ON {COND} WHEN MATCHED THEN DELETE")
    def compact(s):
        s.spark.sql(f"CALL ice.system.rewrite_data_files(table => 'bench.store_sales', options => map('min-input-files','2','target-file-size-bytes','{C.TARGET_FILE_BYTES}'))")
        s.spark.sql("CALL ice.system.rewrite_manifests('bench.store_sales')")
    def expire(s):
        now = dt.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        s.spark.sql(f"CALL ice.system.expire_snapshots(table => 'bench.store_sales', older_than => TIMESTAMP '{now}', retain_last => 1)")
    def history(s):
        rows = s.spark.sql(f"SELECT snapshot_id ref, CAST(unix_millis(committed_at) AS LONG) ts_ms, operation FROM {s.ID}.snapshots ORDER BY committed_at").collect()
        return [r.asDict() for r in rows]
    def audit_fields(s):
        return sorted(s.spark.sql(f"SELECT summary FROM {s.ID}.snapshots LIMIT 1").first()["summary"].keys()) + ["parent_id", "manifest_list", "named tags/branches (refs table)"]
    def tag(s, name): s.spark.sql(f"ALTER TABLE {s.ID} CREATE OR REPLACE TAG `{name}`")
    def ddl(s, op):
        q = {"add": f"ALTER TABLE {s.ID} ADD COLUMN loyalty_tier STRING",
             "rename": f"ALTER TABLE {s.ID} RENAME COLUMN ss_coupon_amt TO ss_coupon_amount",
             "widen": f"ALTER TABLE {s.ID} ALTER COLUMN ss_quantity TYPE BIGINT",
             "drop": f"ALTER TABLE {s.ID} DROP COLUMN ss_ext_tax"}[op]
        s.spark.sql(q)

class Delta(Base):
    def tbl(s): return f"delta.`{s.path}`"
    def write_initial(s, df):
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy(C.PARTITION_COL).save(s.path)
        # 'delta.targetFileSize' is a Databricks-only property (OSS Delta 3.2 rejects it). The target size is set with
        # spark.databricks.delta.optimize.maxFileSize in get_spark() and applies to OPTIMIZE (compaction).
        if C.TABLE_MODE == "mor":
            s.spark.sql(f"ALTER TABLE {s.tbl()} SET TBLPROPERTIES ('delta.enableDeletionVectors'='true')")
    def read(s, ref=None, ts_ms=None):
        r = s.spark.read.format("delta")
        if ref is not None: r = r.option("versionAsOf", ref)
        if ts_ms is not None: r = r.option("timestampAsOf", dt.datetime.utcfromtimestamp(ts_ms / 1000).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3])
        return r.load(s.path)
    def upsert(s, src):
        from delta.tables import DeltaTable
        (DeltaTable.forPath(s.spark, s.path).alias("t").merge(src.alias("s"), COND)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    def delete(s, rows):
        from delta.tables import DeltaTable
        DeltaTable.forPath(s.spark, s.path).alias("t").merge(rows.alias("s"), COND).whenMatchedDelete().execute()
    def compact(s):
        from delta.tables import DeltaTable
        DeltaTable.forPath(s.spark, s.path).optimize().executeCompaction()
    def expire(s):
        from delta.tables import DeltaTable
        DeltaTable.forPath(s.spark, s.path).vacuum(0)   # removes data files; log entries stay but old versions become unreadable
    def history(s):
        rows = s.spark.sql(f"DESCRIBE HISTORY {s.tbl()}").selectExpr("version ref", "CAST(unix_millis(timestamp) AS LONG) ts_ms", "operation").orderBy("version").collect()
        return [r.asDict() for r in rows]
    def audit_fields(s): return s.spark.sql(f"DESCRIBE HISTORY {s.tbl()}").columns
    def tag(s, name): pass   # Delta has no named tags; use userMetadata at write time or store the version number externally
    def ddl(s, op):
        t = s.tbl()
        if op in ("rename", "drop"):
            s.spark.sql(f"ALTER TABLE {t} SET TBLPROPERTIES ('delta.columnMapping.mode'='name','delta.minReaderVersion'='2','delta.minWriterVersion'='5')")
        q = {"add": f"ALTER TABLE {t} ADD COLUMNS (loyalty_tier STRING)",
             "rename": f"ALTER TABLE {t} RENAME COLUMN ss_coupon_amt TO ss_coupon_amount",
             "widen": f"ALTER TABLE {t} ALTER COLUMN ss_quantity TYPE BIGINT",   # needs the type-widening feature; may fail on 3.2
             "drop": f"ALTER TABLE {t} DROP COLUMN ss_ext_tax"}[op]
        s.spark.sql(q)

class Hudi(Base):
    TBL = "hudi_fact"
    def reg(s):
        """Register the path-based table in the (per-process, in-memory) catalog; Hudi procedures such as
        show_commits and run_clean only accept `table =>`, not `path =>`."""
        s.spark.sql(f"CREATE TABLE IF NOT EXISTS {s.TBL} USING hudi LOCATION '{s.path}'"); return s.TBL
    def opts(s, op):
        return {"hoodie.table.name": "store_sales", "hoodie.datasource.write.recordkey.field": ",".join(C.KEY),
                "hoodie.datasource.write.partitionpath.field": C.PARTITION_COL,
                "hoodie.datasource.write.precombine.field": "updated_at",
                "hoodie.datasource.write.keygenerator.class": "org.apache.hudi.keygen.ComplexKeyGenerator",
                "hoodie.datasource.write.table.type": "MERGE_ON_READ" if C.TABLE_MODE == "mor" else "COPY_ON_WRITE",
                "hoodie.datasource.write.operation": op, "hoodie.parquet.max.file.size": str(C.TARGET_FILE_BYTES),
                "hoodie.cleaner.commits.retained": "1000", "hoodie.clean.automatic": "false", "hoodie.metadata.enable": "true",
                "hoodie.upsert.shuffle.parallelism": C.SHUFFLE_PARTS, "hoodie.insert.shuffle.parallelism": C.SHUFFLE_PARTS,
                "hoodie.bulkinsert.shuffle.parallelism": C.SHUFFLE_PARTS, "hoodie.delete.shuffle.parallelism": C.SHUFFLE_PARTS,
                "hoodie.schema.on.read.enable": "true", "hoodie.datasource.write.reconcile.schema": "true"}
    def write_initial(s, df):
        df.write.format("hudi").options(**s.opts("bulk_insert")).mode("overwrite").save(s.path)
    def read(s, ref=None, ts_ms=None):
        r = s.spark.read.format("hudi")
        if ref is not None: r = r.option("as.of.instant", ref)
        if ts_ms is not None: r = r.option("as.of.instant", dt.datetime.utcfromtimestamp(ts_ms / 1000).strftime("%Y%m%d%H%M%S%f")[:-3])
        return r.load(s.path)
    def upsert(s, src): src.write.format("hudi").options(**s.opts("upsert")).mode("append").save(s.path)
    def delete(s, rows): rows.write.format("hudi").options(**s.opts("delete")).mode("append").save(s.path)
    def compact(s):
        # COW: clustering is Hudi's file-compaction equivalent. MOR: run_compaction as well.
        # Procedure signatures vary slightly between Hudi releases -> verify against your version's docs.
        if C.TABLE_MODE == "mor": s.spark.sql(f"call run_compaction(op => 'scheduleandexecute', path => '{s.path}')")
        s.spark.sql(f"call run_clustering(path => '{s.path}', op => 'scheduleandexecute', options => 'hoodie.clean.automatic=false')")
    def expire(s):
        s.spark.sql(f"call run_clean(table => '{s.reg()}', clean_policy => 'KEEP_LATEST_COMMITS', retain_commits => 1)")
    def history(s):
        rows = s.spark.sql(f"call show_commits(table => '{s.reg()}', limit => 10000)").collect()
        out = [{"ref": r["commit_time"], "ts_ms": int(dt.datetime.strptime(r["commit_time"][:17], "%Y%m%d%H%M%S%f").replace(tzinfo=dt.timezone.utc).timestamp() * 1000), "operation": "commit"} for r in rows]
        return sorted(out, key=lambda x: x["ref"])
    def audit_fields(s): return s.spark.sql(f"call show_commits(table => '{s.reg()}', limit => 1)").columns + ["extraMetadata", "savepoints"]
    def tag(s, name): pass   # Hudi savepoints pin a commit but are not named tags
    def ddl(s, op):
        s.reg()
        s.spark.sql("SET hoodie.schema.on.read.enable=true")
        q = {"add": "ALTER TABLE hudi_fact ADD COLUMNS (loyalty_tier STRING)",
             "rename": "ALTER TABLE hudi_fact RENAME COLUMN ss_coupon_amt TO ss_coupon_amount",
             "widen": "ALTER TABLE hudi_fact ALTER COLUMN ss_quantity TYPE BIGINT",
             "drop": "ALTER TABLE hudi_fact DROP COLUMN ss_ext_tax"}[op]
        s.spark.sql(q)

def get_table(spark, fmt): return {"iceberg": Iceberg, "delta": Delta, "hudi": Hudi}[fmt](spark, fmt)

def cli():
    ap = argparse.ArgumentParser(); ap.add_argument("--format", required=True, choices=C.FORMATS); return ap.parse_args()

def make_batch(tbl, total_rows, fraction, scenario, seed, offset):
    """Half updates (existing keys, changed price) + half inserts (new keys). Materialised before timing."""
    base = tbl.read()
    base = base.drop(*[c for c in base.columns if c.startswith("_hoodie_")])
    if scenario == "recent":
        top = sorted(r[0] for r in base.select(C.PARTITION_COL).distinct().collect())[-2:]
        base = base.where(F.col(C.PARTITION_COL).isin(top))
    n = max(base.count(), 1)
    half = min(1.0, (fraction * total_rows / 2) / n)
    upd = base.sample(False, half, seed).withColumn("ss_net_paid", F.round(F.col("ss_net_paid") * 1.01, 2)).withColumn("updated_at", F.current_timestamp())
    ins = (base.sample(False, half, seed + 1).withColumn("ss_ticket_number", F.col("ss_ticket_number") + F.lit(10_000_000_000 + offset))
               .withColumn("updated_at", F.current_timestamp()))
    b = upd.unionByName(ins).persist(); return b, b.count()

In [ ]:
%%writefile /content/lakehouse-bench/scripts/00_generate_data.py
"""Generate TPC-DS parquet with DuckDB's dsdgen. store_sales spans 1998-2003 (~5 years of order lines).
For SF >= 100 prefer the native TPC-DS dsdgen with -PARALLEL/-CHILD chunks, convert to parquet, and put it in config.DATA_DIR.
NYC TLC alternative: download monthly yellow_tripdata_YYYY-MM.parquet (5 years) and adapt KEY/PARTITION_COL in config.py."""
import os, sys, duckdb
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import config as C

TABLES = ["store_sales", "date_dim", "item", "customer", "store"]
os.makedirs(C.DATA_DIR, exist_ok=True)
con = duckdb.connect()
con.execute("INSTALL tpcds; LOAD tpcds;")
con.execute(f"CALL dsdgen(sf={C.SF})")
for t in TABLES:
    out = f"{C.DATA_DIR}/{t}"
    os.makedirs(out, exist_ok=True)
    con.execute(f"COPY {t} TO '{out}' (FORMAT PARQUET, PER_THREAD_OUTPUT TRUE, COMPRESSION SNAPPY)")
    print("wrote", out, con.execute(f"SELECT count(*) FROM {t}").fetchone()[0], "rows")

In [ ]:
%%writefile /content/lakehouse-bench/scripts/01_load.py
"""Load 5 years of store_sales into one format. Records load time, size, file count, baseline parquet size."""
from common import *
fmt = cli().format; spark = get_spark(fmt); t = get_table(spark, fmt)
ss = spark.read.parquet(f"{C.DATA_DIR}/store_sales")
dd = spark.read.parquet(f"{C.DATA_DIR}/date_dim").select("d_date_sk", F.date_format("d_date", "yyyy-MM").alias(C.PARTITION_COL))
df = (ss.join(F.broadcast(dd), ss.ss_sold_date_sk == dd.d_date_sk, "left").drop("d_date_sk")
        .withColumn(C.PARTITION_COL, F.coalesce(F.col(C.PARTITION_COL), F.lit("0000-00")))
        .withColumn("updated_at", F.current_timestamp()))
src_rows = ss.count()
with timed() as r: t.write_initial(df)
h = t.history()
n = t.read().count()
state_put(fmt, initial_rows=n, v0=h[0]["ref"], v0_ts=h[0]["ts_ms"])
record(fmt, "load", "load_seconds", r["seconds"], "s", rows=n, source_rows=src_rows)
record(fmt, "load", "rows_per_second", round(n / r["seconds"]), "rows/s")
record(fmt, "load", "table_bytes", t.size(), "bytes", data_files=t.files(), metadata_bytes=t.meta())
record(fmt, "load", "source_parquet_bytes", dir_bytes(spark, f"{C.DATA_DIR}/store_sales"), "bytes")

In [ ]:
%%writefile /content/lakehouse-bench/scripts/02_queries.py
"""Query latency: 1 cold + N warm runs per query. Dimensions are plain parquet for every format, only the fact table differs."""
import statistics
from common import *
fmt = cli().format; spark = get_spark(fmt); t = get_table(spark, fmt)
for d in ("date_dim", "item", "customer"): spark.read.parquet(f"{C.DATA_DIR}/{d}").createOrReplaceTempView(d)
t.view("fact")
pick = t.read().where("sold_ym='2001-06'").select(*C.KEY).limit(1).first()
Q = {
 "q1_full_scan_agg": "SELECT d.d_year, sum(f.ss_net_paid) s FROM fact f JOIN date_dim d ON f.ss_sold_date_sk=d.d_date_sk GROUP BY d.d_year ORDER BY 1",
 "q2_partition_prune": "SELECT count(*) c, sum(ss_net_paid) s FROM fact WHERE sold_ym='2001-06'",
 "q3_point_lookup": f"SELECT * FROM fact WHERE ss_item_sk={pick[0]} AND ss_ticket_number={pick[1]}",
 "q4_star_join_tpcds_q3": """SELECT d.d_year, i.i_brand_id, i.i_brand, sum(f.ss_ext_sales_price) s FROM fact f
        JOIN date_dim d ON f.ss_sold_date_sk=d.d_date_sk JOIN item i ON f.ss_item_sk=i.i_item_sk
        WHERE i.i_manufact_id=128 AND d.d_moy=11 GROUP BY 1,2,3 ORDER BY 1,4 DESC,2 LIMIT 100""",
 "q5_top_customers": "SELECT ss_customer_sk, sum(ss_net_paid) s FROM fact WHERE ss_customer_sk IS NOT NULL GROUP BY 1 ORDER BY s DESC LIMIT 100",
}
for name, sql in Q.items():
    runs = []
    for i in range(1 + C.WARM_RUNS):
        with timed() as r: spark.sql(sql).collect()
        runs.append(r["seconds"])
    record(fmt, "queries", f"{name}_cold", runs[0], "s")
    record(fmt, "queries", f"{name}_warm_median", statistics.median(runs[1:]), "s", runs=runs)

In [ ]:
%%writefile /content/lakehouse-bench/scripts/03_upserts.py
"""Upsert (MERGE) latency + bytes written for 0.1 / 1 / 5 % batches, 'recent' vs 'scattered' key distributions."""
from common import *
fmt = cli().format; spark = get_spark(fmt); t = get_table(spark, fmt)
total = state_get(fmt)["initial_rows"]; k = 0
for scen in C.UPSERT_SCENARIOS:
    for frac in C.UPSERT_FRACTIONS:
        k += 1; batch, rows = make_batch(t, total, frac, scen, seed=100 + k, offset=k * 1_000_000)
        size0, files0 = t.size(), t.files()
        with timed() as r: t.upsert(batch)
        tag = f"{scen}_{frac*100:g}pct"
        record(fmt, "upserts", f"{tag}_seconds", r["seconds"], "s", batch_rows=rows)
        record(fmt, "upserts", f"{tag}_rows_per_second", round(rows / r["seconds"]), "rows/s")
        record(fmt, "upserts", f"{tag}_bytes_added", t.size() - size0, "bytes", files_added=t.files() - files0)
        batch.unpersist()

In [ ]:
%%writefile /content/lakehouse-bench/scripts/04_compaction.py
"""Create small-file fragmentation with many tiny commits, then time compaction and re-measure a pruned query."""
from common import *
import statistics
fmt = cli().format; spark = get_spark(fmt); t = get_table(spark, fmt)
total = state_get(fmt)["initial_rows"]
for i in range(C.FRAGMENT_BATCHES):
    b, _ = make_batch(t, total, 0.0002, "recent", seed=500 + i, offset=(900 + i) * 1_000_000); t.upsert(b); b.unpersist()
def q2():
    t.view("fact"); xs = []
    for _ in range(4):
        with timed() as r: spark.sql("SELECT count(*), sum(ss_net_paid) FROM fact WHERE sold_ym >= '2002-01' AND sold_ym <= '2002-06'").collect()
        xs.append(r["seconds"])
    return statistics.median(xs[1:])
f0, b0, q0 = t.files(), t.size(), q2()
with timed() as r: t.compact()
f1, b1, q1 = t.files(), t.size(), q2()
record(fmt, "compaction", "compaction_seconds", r["seconds"], "s")
record(fmt, "compaction", "files_before", f0, "files"); record(fmt, "compaction", "files_after", f1, "files")
record(fmt, "compaction", "bytes_before", b0, "bytes"); record(fmt, "compaction", "bytes_after", b1, "bytes")
record(fmt, "compaction", "pruned_query_before", q0, "s"); record(fmt, "compaction", "pruned_query_after", q1, "s")

In [ ]:
%%writefile /content/lakehouse-bench/scripts/06_time_travel_schema.py
"""Functional + latency tests: time travel by version and timestamp, then schema evolution (add / rename / widen / drop)."""
from common import *
fmt = cli().format; spark = get_spark(fmt); t = get_table(spark, fmt); st = state_get(fmt)
h = t.history()
# --- time travel ---
with timed() as r: n_ver = t.read(ref=st["v0"]).count()
record(fmt, "timetravel", "by_version_ok", int(n_ver == st["initial_rows"]), "bool", rows=n_ver, expected=st["initial_rows"])
record(fmt, "timetravel", "by_version_seconds", r["seconds"], "s")
mid = h[len(h) // 2]
with timed() as r: n_ts = t.read(ts_ms=mid["ts_ms"]).count()
record(fmt, "timetravel", "by_timestamp_seconds", r["seconds"], "s", rows=n_ts, snapshots_listed=len(h))
record(fmt, "timetravel", "history_entries", len(h), "versions")
# --- schema evolution (each op isolated; failures are results, not crashes) ---
cols_before = t.read().columns
for op in ("add", "rename", "widen", "drop"):
    try:
        f0 = t.files(); b0 = t.size()
        with timed() as r: t.ddl(op)
        ok, err = 1, ""
    except Exception as e:
        ok, err, r = 0, str(e)[:300], {"seconds": None}
    record(fmt, "schema", f"{op}_ok", ok, "bool", error=err, seconds=r["seconds"], data_rewritten=bool(ok and t.size() > b0 * 1.05))
# old data still readable with the old schema?
try:
    old_cols = t.read(ref=st["v0"]).columns
    record(fmt, "schema", "old_version_returns_old_schema", int(old_cols == cols_before), "bool", old=len(old_cols), now=len(t.read().columns))
except Exception as e:
    record(fmt, "schema", "old_version_returns_old_schema", 0, "bool", error=str(e)[:300])

In [ ]:
%%writefile /content/lakehouse-bench/scripts/07_audit_repro.py
"""Reproducibility of a past ML training set. Fingerprint a training window, mutate the table (upsert, delete, compaction),
re-read the old snapshot by version AND by timestamp, then expire history and see what survives.
Every read/maintenance step is isolated: a failure is recorded as a result (value 0 + error text), never a crash."""
from common import *
fmt = cli().format; spark = get_spark(fmt); t = get_table(spark, fmt); total = state_get(fmt)["initial_rows"]
W = "sold_ym BETWEEN '1999-01' AND '2001-12'"           # the 'training window'
def err(e): return str(e).splitlines()[0][:300] if str(e) else repr(e)[:300]
def try_fp(label, **tt):
    try: return fingerprint(t.read(**tt).where(W)), ""
    except Exception as e: print(f"[{fmt}] {label} failed: {err(e)}"); return None, err(e)
def try_do(label, fn):
    try: fn(); return ""
    except Exception as e: print(f"[{fmt}] {label} failed: {err(e)}"); return err(e)
ref0 = t.history()[-1]; fp0 = fingerprint(t.read(ref=ref0["ref"]).where(W))
tag_err = try_do("tag", lambda: t.tag("training_run_2026_01"))   # no-op where unsupported
record(fmt, "audit", "fingerprint_t0", fp0["rows"], "rows", tag_error=tag_err, **fp0)
b, _ = make_batch(t, total, 0.01, "scattered", seed=900, offset=77_000_000); t.upsert(b)
d, _ = make_batch(t, total, 0.001, "scattered", seed=901, offset=0)
t.delete(d.where(F.col("ss_ticket_number") < 10_000_000_000)); b.unpersist(); d.unpersist()
compact_err = try_do("compaction", t.compact)
now, e_now = try_fp("current read")
record(fmt, "audit", "current_differs_from_t0", int(now is not None and now != fp0), "bool", now=now, error=e_now, compaction_error=compact_err)
by_ver, e_ver = try_fp("read by version", ref=ref0["ref"])
by_ts, e_ts = try_fp("read by timestamp", ts_ms=ref0["ts_ms"])
record(fmt, "audit", "reproduced_by_version", int(by_ver == fp0), "bool", error=e_ver)
record(fmt, "audit", "reproduced_by_timestamp", int(by_ts == fp0), "bool", error=e_ts)
try: fields = t.audit_fields()
except Exception as e: fields = []; print(f"[{fmt}] audit_fields failed: {err(e)}")
record(fmt, "audit", "audit_fields", len(fields), "fields", fields=fields)
exp_err = try_do("expire", t.expire)
after, e_after = try_fp("read after expire", ref=ref0["ref"])
record(fmt, "audit", "reproducible_after_expire_default", int(after == fp0), "bool", error=e_after or exp_err,
       note="unpinned history is lost on expiry; Iceberg tag (taken at t0) should keep the snapshot alive")


In [ ]:
%%writefile /content/lakehouse-bench/scripts/05_storage.py
"""Storage cost after the whole run, before/after expiring history. Cost model = bytes * S3 price (config)."""
from common import *
fmt = cli().format; spark = get_spark(fmt); t = get_table(spark, fmt)
gb = lambda x: x / 1024**3
b0, m0, src = t.size(), t.meta(), dir_bytes(spark, f"{C.DATA_DIR}/store_sales")
with timed() as r: t.expire()
b1 = t.size()
record(fmt, "storage", "bytes_with_history", b0, "bytes", metadata_bytes=m0)
record(fmt, "storage", "bytes_after_expire", b1, "bytes", expire_seconds=r["seconds"])
record(fmt, "storage", "overhead_vs_source_parquet", round(b1 / src, 3), "x")
record(fmt, "storage", "usd_per_month_with_history", round(gb(b0) * C.S3_USD_PER_GB_MONTH, 4), "USD")
record(fmt, "storage", "usd_per_month_after_expire", round(gb(b1) * C.S3_USD_PER_GB_MONTH, 4), "USD")

In [ ]:
%%writefile /content/lakehouse-bench/scripts/aggregate.py
"""Merge results/raw/*.jsonl -> results/summary.json, results/comparison.csv and site/results.js (read by the web report)."""
import glob, json, os, csv, collections
root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
rows = [json.loads(l) for f in glob.glob(f"{root}/results/raw/*.jsonl") for l in open(f)]
latest = {}
for r in rows: latest[(r["format"], r["test"], r["metric"])] = r          # last write wins
tbl = collections.defaultdict(dict)
for (fmt, test, m), r in latest.items(): tbl[f"{test}.{m}"][fmt] = {"value": r["value"], "unit": r["unit"], "detail": r["detail"]}
out = {"metrics": tbl, "runs": len(rows), "sf": rows[0]["sf"] if rows else None, "mode": rows[0]["mode"] if rows else None,
       "versions": rows[0]["versions"] if rows else None}
json.dump(out, open(f"{root}/results/summary.json", "w"), indent=1, default=str)
with open(f"{root}/results/comparison.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["metric", "unit", "iceberg", "delta", "hudi"])
    for k, v in sorted(tbl.items()): w.writerow([k, next(iter(v.values()))["unit"]] + [v.get(x, {}).get("value", "") for x in ("iceberg", "delta", "hudi")])
open(f"{root}/site/results.js", "w").write("window.RESULTS = " + json.dumps(out, default=str) + ";")
print("metrics:", len(tbl), "| rows:", len(rows))

## 3. Settings

In [ ]:
SF = 1                      # TPC-DS scale factor. 1 = free Colab. 10 needs High-RAM and more disk.
TABLE_MODE = "cow"          # "cow" or "mor" (merge-on-read / deletion vectors where supported)
FORMATS_TO_RUN = ["iceberg", "delta", "hudi"]
RESET = False               # True = wipe tables, results and step cache (keeps generated source data) for a clean re-run
os.environ.update({"TPCDS_SF": str(SF), "BENCH_ROOT": "/content/bench_data", "TABLE_MODE": TABLE_MODE, "SPARK_MASTER": "local[*]",
                   "DRIVER_MEM": "8g", "SHUFFLE_PARTS": "8"})
if RESET:
    import shutil
    shutil.rmtree(f"/content/bench_data/tpcds_sf{SF}/warehouse", ignore_errors=True)
    for p in glob.glob(f"{ROOT}/results/raw/*") + glob.glob(f"{ROOT}/results/state_*.json") + [f"{ROOT}/results/status.json"]:
        os.remove(p) if os.path.isfile(p) else None
    print("RESET: warehouse, results and step cache cleared")
print("SF", SF, "| mode", TABLE_MODE, "| formats", FORMATS_TO_RUN)

## 4. Run everything
Each step is logged to `results/logs/`. A failed step is shown with its error lines and the run continues. If a format fails at *load*, its remaining steps are skipped. Re-running this cell skips steps that already succeeded.

In [ ]:
import subprocess, time, re, json, pandas as pd
os.makedirs(f"{ROOT}/results/logs", exist_ok=True)
STEPS = ["01_load", "02_queries", "03_upserts", "04_compaction", "06_time_travel_schema", "07_audit_repro", "05_storage"]
SP = f"{ROOT}/results/status.json"; DONE = json.load(open(SP)) if os.path.exists(SP) else {}
for _f in FORMATS_TO_RUN:                               # a format whose load failed earlier is reloaded from scratch
    if DONE.get(f"{_f}:01_load") == "FAIL":
        for _s in STEPS: DONE.pop(f"{_f}:{_s}", None)
        for _p in glob.glob(f"{ROOT}/results/raw/*_{_f}.jsonl"): os.remove(_p)
def run_step(script, fmt=None):
    cmd = [PY, f"scripts/{script}.py"] + (["--format", fmt] if fmt else [])
    t = time.time(); p = subprocess.run(cmd, cwd=ROOT, env=os.environ, capture_output=True, text=True)
    out = p.stdout + p.stderr; open(f"{ROOT}/results/logs/{fmt or 'data'}_{script}.log", "w").write(out)
    return p.returncode, round(time.time() - t), out
def errtail(out):
    l = [x for x in out.splitlines() if re.search(r"Error|Exception|Caused by|Traceback|not found|Failed|AnalysisException", x)]
    return "\n".join(l[-10:]) or out[-700:]
STATUS_ROWS = []
def do(script, fmt=None):
    key = f"{fmt}:{script}"
    if DONE.get(key) == "OK": print(f"SKIP  {key} (already done)"); STATUS_ROWS.append((fmt or "-", script, "OK (cached)", 0)); return True
    print(f"RUN   {key} ...", flush=True); rc, sec, out = run_step(script, fmt)
    ok = rc == 0; DONE[key] = "OK" if ok else "FAIL"; json.dump(DONE, open(SP, "w"))
    print(f"{'OK  ' if ok else 'FAIL'}  {key}  ({sec}s)"); STATUS_ROWS.append((fmt or "-", script, "OK" if ok else "FAIL", sec))
    if not ok: print(errtail(out)); print(f"   full log: results/logs/{fmt or 'data'}_{script}.log")
    return ok
if do("00_generate_data"):
    for f in FORMATS_TO_RUN:
        for s in STEPS:
            ok = do(s, f)
            if not ok and s == "01_load": print(f"!! {f}: load failed, skipping its remaining steps"); break
rc, sec, out = run_step("aggregate"); print(out.strip())
STATUS = pd.DataFrame(STATUS_ROWS, columns=["format", "step", "status", "seconds"]); STATUS

In [ ]:
#@title (only if a step FAILED) print its full log
FAILED_KEY = "iceberg_01_load"   # e.g. delta_03_upserts
print(open(f"{ROOT}/results/logs/{FAILED_KEY}.log").read()[-6000:])

## 5. Detailed results

### 5.1 Load the data

In [ ]:
import glob, json, base64, io, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
FM = ["iceberg", "delta", "hudi"]; COL = {"iceberg": "#FFD600", "delta": "#60A5FA", "hudi": "#FF6B35"}
rows = [json.loads(l) for f in glob.glob(f"{ROOT}/results/raw/*.jsonl") for l in open(f)]
assert rows, "No results were produced: scroll up to the run log and read the FAIL details first."
raw = pd.DataFrame(rows).drop_duplicates(["format", "test", "metric"], keep="last")
raw["value_n"] = pd.to_numeric(raw["value"], errors="coerce")
META = rows[0]; print(f"{len(raw)} measurements | formats: {sorted(raw.format.unique())} | SF={META['sf']} | mode={META['mode']}")
def table(test, pat=None):
    d = raw[raw.test == test]
    if pat: d = d[d.metric.str.contains(pat)]
    pv = d.pivot(index="metric", columns="format", values="value_n")
    pv = pv.reindex(columns=[f for f in FM if f in pv.columns])
    pv.insert(0, "unit", d.drop_duplicates("metric").set_index("metric")["unit"].reindex(pv.index)); return pv
HB = lambda m: any(k in m for k in ("rows_per_second", "_ok", "reproduced", "history_entries", "audit_fields", "returns_old"))
def hl(r):
    v = pd.to_numeric(r, errors="coerce"); t = v.max() if HB(r.name) else v.min()
    return ["background:#665500;font-weight:700" if (pd.notna(x) and x == t and v.notna().sum() > 1) else "" for x in v]
SEC = {}
TITLES = [("load", "1. Load"), ("queries", "2. Query latency (seconds, 1 cold + warm median)"), ("upserts", "3. Upserts (MERGE)"), ("compaction", "4. Compaction"),
          ("storage", "5. Storage & cost"), ("timetravel", "6. Time travel"), ("schema", "7. Schema evolution (1 = pass)"), ("audit", "8. Audit / reproducibility")]

### 5.2 Result tables (best value highlighted)

In [ ]:
for t, title in TITLES:
    if (raw.test == t).any():
        print(title); pv = table(t); SEC[t] = pv; nc = [c for c in pv.columns if c != "unit"]
        display(pv.style.format({c: "{:,.3f}" for c in nc}, na_rep="—").apply(hl, axis=1, subset=nc))
print("Highlighted = best value in the row (lowest, or highest for rate / pass-fail metrics).")

### 5.3 Charts

In [ ]:
CH = {}
def chart(key, test, pat, title, xl):
    d = raw[raw.test == test]
    if not len(d): return
    pv = table(test, pat).drop(columns="unit")
    if pv.empty: return
    ax = pv.plot.barh(figsize=(9, max(2.5, .42 * len(pv) + 1)), color=[COL[c] for c in pv.columns], edgecolor="black")
    ax.set_title(title); ax.set_xlabel(xl); ax.invert_yaxis(); plt.tight_layout()
    b = io.BytesIO(); plt.savefig(b, format="png", dpi=120); CH[key] = base64.b64encode(b.getvalue()).decode(); plt.show()
chart("q", "queries", "warm_median", "Query latency (warm median)", "seconds, lower is better")
chart("u", "upserts", "_seconds$", "Upsert time per batch", "seconds, lower is better")
chart("c", "compaction", "seconds|pruned", "Compaction time and pruned-scan latency", "seconds, lower is better")
chart("s", "storage", "^bytes", "Table size on disk", "bytes, lower is better")

### 5.4 Failed tests, error text and audit fields

In [ ]:
f = raw[(raw.test.isin(["schema", "timetravel", "audit"])) & (raw.value_n == 0)].copy()
f["error"] = f["detail"].apply(lambda d: (d or {}).get("error", "") if isinstance(d, dict) else "")
print("Tests that FAILED (value 0). These are results, not bugs, unless the error text shows a script problem:")
FAILS = f[["format", "test", "metric", "error"]].reset_index(drop=True); display(FAILS)
aud = raw[(raw.test == "audit") & (raw.metric == "audit_fields")][["format", "detail"]]
for _, r in aud.iterrows(): print(f"\nAudit fields exposed by {r.format}:", ", ".join(r.detail.get("fields", [])))

### 5.5 Weighted scorecard
The rubric (documented capabilities) is blended with measured results where available. Edit `W` and re-run to change the weights.

In [ ]:
RUB = {"audit": {"iceberg": 5, "delta": 4, "hudi": 3}, "schema": {"iceberg": 5, "delta": 3, "hudi": 3}, "upsert": {"iceberg": 3, "delta": 4, "hudi": 5},
       "query": {"iceberg": 4, "delta": 4, "hudi": 3}, "storage": {"iceberg": 4, "delta": 4, "hudi": 3}, "eco": {"iceberg": 5, "delta": 4, "hudi": 3}, "ops": {"iceberg": 3, "delta": 5, "hudi": 2}}
W = {"audit": 8, "schema": 6, "upsert": 5, "query": 6, "storage": 4, "eco": 6, "ops": 4}   # audit-first weights for this case study: edit and re-run
def val(t, m, f):
    x = raw[(raw.test == t) & (raw.metric == m) & (raw.format == f)].value_n
    return float(x.iloc[0]) if len(x) and pd.notna(x.iloc[0]) else None
def ratio(t, pred, f):
    out = []
    for m in [m for m in raw[raw.test == t].metric.unique() if pred(m)]:
        vs = {g: val(t, m, g) for g in FM}; vv = [x for x in vs.values() if x]
        if vs[f] and vv: out.append(min(vv) / vs[f])
    return sum(out) / len(out) if out else None
def crit(c, f):
    if c == "upsert":  r = ratio("upserts", lambda m: m.endswith("_seconds"), f);  return (5 * r, "measured") if r else (RUB[c][f], "rubric")
    if c == "query":   r = ratio("queries", lambda m: m.endswith("warm_median"), f); return (5 * r, "measured") if r else (RUB[c][f], "rubric")
    if c == "storage": r = ratio("storage", lambda m: m == "bytes_after_expire", f);  return (5 * r, "measured") if r else (RUB[c][f], "rubric")
    if c == "audit":
        a = [val("audit", k, f) for k in ("reproduced_by_version", "reproduced_by_timestamp")]
        return ((RUB[c][f] + 5 * sum(a) / 2) / 2, "rubric+measured") if None not in a else (RUB[c][f], "rubric")
    if c == "schema":
        a = [val("schema", o + "_ok", f) for o in ("add", "rename", "widen", "drop")]
        return ((RUB[c][f] + 5 * sum(a) / 4) / 2, "rubric+measured") if None not in a else (RUB[c][f], "rubric")
    return (RUB[c][f], "rubric")
fm = [f for f in FM if f in raw.format.unique()]
CS = pd.DataFrame({f: {c: crit(c, f)[0] for c in RUB} for f in fm}); SRC = {c: crit(c, fm[0])[1] for c in RUB}
TOT = {f: 100 * sum(W[c] * CS.loc[c, f] / 5 for c in RUB) / sum(W.values()) for f in fm}
CS["weight"] = pd.Series(W); CS["source"] = pd.Series(SRC); display(CS.round(2))
SCORE = pd.Series(TOT).sort_values(ascending=False).round(1); print("Weighted score (0-100):"); display(SCORE.to_frame("score"))
ax = SCORE.plot.bar(color=[COL[f] for f in SCORE.index], edgecolor="black", figsize=(5, 3)); ax.set_ylim(0, 100); plt.tight_layout()
b = io.BytesIO(); plt.savefig(b, format="png", dpi=120); CH["score"] = base64.b64encode(b.getvalue()).decode(); plt.show()

### 5.6 Recommendation (generated from the numbers)

In [ ]:
win = SCORE.index[0]; run = SCORE.index[1] if len(SCORE) > 1 else None
def lead(t, pred):
    s = {f: ratio(t, pred, f) for f in fm}; s = {k: v for k, v in s.items() if v}; return max(s, key=s.get) if s else None
L = {"query": lead("queries", lambda m: m.endswith("warm_median")), "upsert": lead("upserts", lambda m: m.endswith("_seconds")), "storage": lead("storage", lambda m: m == "bytes_after_expire")}
slow = [f"{f} ({ratio(t, p, f):.2f}x of best speed)" for f in fm for t, p in (("upserts", lambda m: m.endswith("_seconds")), ("queries", lambda m: m.endswith("warm_median"))) if ratio(t, p, f) and ratio(t, p, f) < .5]
def passed(t, ms, f): return sum(int(val(t, m, f) or 0) for m in ms)
aud_pass = {f: passed("audit", ["reproduced_by_version", "reproduced_by_timestamp"], f) for f in fm}
sch_pass = {f: passed("schema", [o + "_ok" for o in ("add", "rename", "widen", "drop")], f) for f in fm}
exp = {f: val("audit", "reproducible_after_expire_default", f) for f in fm}
REC = f"""## Recommendation (generated from the measured results)

**Recommended format: {win.upper()}** (weighted score {SCORE[win]}/100{f", ahead of {run.upper()} at {SCORE[run]}" if run else ""}), using audit-first weights {W}.

**Measured leaders:** queries: {L['query']}; upserts: {L['upsert']}; smallest storage after expiry: {L['storage']}.
**Audit replay (by version + timestamp, max 2):** {aud_pass}. **Training set still reproducible after default expiry:** {exp} (1 = yes).
**Schema operations passed (max 4):** {sch_pass}.
**Override rule (>2x slower than best on queries/upserts):** {"; ".join(slow) if slow else "no format breached the 2x threshold"}.
**Failed tests:** {len(FAILS)} (see the failures table; a failed feature is a finding about that format and release).

**Caveats:** single-node Spark, scale factor {META['sf']}, {META['mode'].upper()} tables, one run per test. Relative gaps matter more than absolute seconds. Re-run at a larger scale factor on object storage before a production decision.
**Policy regardless of format:** retain snapshots longer than the model-audit window and pin or record the version of every training run.
"""
display(Markdown(REC)); open(f"{ROOT}/results/recommendation.md", "w").write(REC)

### 5.7 Build the downloadable report

In [ ]:
import html as H
env = open(f"{ROOT}/results/env.txt").read() if os.path.exists(f"{ROOT}/results/env.txt") else ""
parts = [f"<h1>Iceberg vs Delta Lake vs Hudi: benchmark report</h1><pre>{H.escape(env)}\nSF={META['sf']} mode={META['mode']} versions={META['versions']}</pre>"]
try: parts.append("<h2>Run log</h2>" + STATUS.to_html(index=False))
except NameError: pass
for t, title in TITLES:
    if t in SEC: parts.append(f"<h2>{title}</h2>" + SEC[t].round(3).to_html(na_rep="—"))
for k, n in (("q", "Query latency"), ("u", "Upserts"), ("c", "Compaction"), ("s", "Storage"), ("score", "Weighted score")):
    if k in CH: parts.append(f"<h3>{n}</h3><img src='data:image/png;base64,{CH[k]}'>")
parts += ["<h2>Failed tests</h2>" + FAILS.to_html(index=False), "<h2>Scorecard</h2>" + CS.round(2).to_html(), "<pre>" + H.escape(REC) + "</pre>"]
css = "body{background:#0A0A0A;color:#F5F5F0;font:14px 'IBM Plex Mono',monospace;padding:32px}table{border-collapse:collapse;margin:12px 0}td,th{border:1px solid #2D2D2D;padding:6px 10px}th{background:#111;color:#FFD600}h1,h2,h3{color:#FFD600}img{max-width:100%;background:#fff}pre{white-space:pre-wrap}"
open(f"{ROOT}/results/report.html", "w").write(f"<html><head><meta charset='utf-8'><style>{css}</style></head><body>" + "".join(parts) + "</body></html>")
print("report.html written")

## 6. Download everything

In [ ]:
!cd /content && zip -qr lakehouse_results_bundle.zip lakehouse-bench/results lakehouse-bench/scripts lakehouse-bench/config.py
from google.colab import files
files.download(f"{ROOT}/results/report.html"); files.download("/content/lakehouse_results_bundle.zip")